In [19]:
from __future__ import annotations

import logging
import math
from datetime import datetime
from typing import Optional, Tuple, Union

import duckdb
import geopandas as gpd
import numpy as np
import pandas as pd
import plotly.express as px
from datasets import load_dataset
from dateutil import rrule
from keplergl import KeplerGl
from shapely import wkt
from shapely.geometry import Polygon
from mobile_coverage.common.logging import configure_logger
from mobile_coverage.common import config
from mobile_coverage.data.load import get_data

log = configure_logger("cell_coverage.sector_polygons")

In [20]:
def get_centroid(poly_wkt: str) -> Optional[Tuple[float, float]]:
    """
    Compute the centroid (x, y) of a given WKT polygon.

    Args:
        poly_wkt: Polygon in WKT format.

    Returns:
        Tuple of (centroid_x, centroid_y), or None if invalid.
    """
    if not isinstance(poly_wkt, str):
        return None

    try:
        poly = wkt.loads(poly_wkt)
        return poly.centroid.x, poly.centroid.y
    except Exception as exc:
        log.debug("Error processing WKT: %s | Error: %s", poly_wkt, exc)
        return None

In [21]:
def infer_location(
    lat: float,
    lon: float,
    distance_m: float,
    bearing_deg: float,
    round_dp: Optional[int] = 4,
) -> Tuple[float, float]:
    """
    Infer a new lon/lat given a starting point, distance, and bearing.

    Args:
        lat: Starting latitude in degrees.
        lon: Starting longitude in degrees.
        distance_m: Distance to travel in metres.
        bearing_deg: Bearing in degrees clockwise from North.
        round_dp: Optional decimal precision for output coordinates.

    Returns:
        Tuple of (longitude, latitude).
    """
    earth_radius_km = 6378.1
    bearing_rad = math.radians(bearing_deg)
    distance_km = distance_m / 1000.0

    lat1 = math.radians(lat)
    lon1 = math.radians(lon)

    lat2 = math.asin(
        math.sin(lat1) * math.cos(distance_km / earth_radius_km)
        + math.cos(lat1) * math.sin(distance_km / earth_radius_km) * math.cos(bearing_rad)
    )

    lon2 = lon1 + math.atan2(
        math.sin(bearing_rad) * math.sin(distance_km / earth_radius_km) * math.cos(lat1),
        math.cos(distance_km / earth_radius_km) - math.sin(lat1) * math.sin(lat2),
    )

    out_lon = math.degrees(lon2)
    out_lat = math.degrees(lat2)

    if round_dp is not None:
        out_lon = round(out_lon, round_dp)
        out_lat = round(out_lat, round_dp)

    return out_lon, out_lat

In [22]:
def sector_polygon(
    lat: float,
    lon: float,
    radius_m: float,
    azimuth: float,
    horizontal_beam: float,
    n_arc_points: int = 21,
    wkt_output: bool = True,
    round_dp: Optional[int] = 4,
) -> Optional[Union[Polygon, str]]:
    """
    Build a sector / wedge polygon from cell metadata.

    Args:
        lat: Centre latitude.
        lon: Centre longitude.
        radius_m: Sector radius in metres.
        azimuth: Sector azimuth in degrees clockwise from North.
        horizontal_beam: Total beamwidth in degrees.
        n_arc_points: Number of points to generate across the arc.
        wkt_output: Whether to return WKT instead of a Shapely Polygon.
        round_dp: Optional decimal precision for output coordinates.

    Returns:
        Sector polygon as WKT or Polygon, or None if invalid.
    """
    if any(pd.isna(v) for v in (lat, lon, radius_m, azimuth, horizontal_beam)):
        return None

    if radius_m <= 0 or horizontal_beam <= 0:
        return None

    if n_arc_points < 3:
        n_arc_points = 3

    start_bearing = azimuth - (horizontal_beam / 2.0)
    end_bearing = azimuth + (horizontal_beam / 2.0)

    centre_lon = round(lon, round_dp) if round_dp is not None else lon
    centre_lat = round(lat, round_dp) if round_dp is not None else lat

    points = [(centre_lon, centre_lat)]

    for i in range(n_arc_points):
        frac = i / (n_arc_points - 1)
        bearing = start_bearing + (end_bearing - start_bearing) * frac
        pt_lon, pt_lat = infer_location(
            lat=lat,
            lon=lon,
            distance_m=radius_m,
            bearing_deg=bearing,
            round_dp=round_dp,
        )
        points.append((pt_lon, pt_lat))

    points.append((centre_lon, centre_lat))

    try:
        polygon = Polygon(points)
        if not polygon.is_valid:
            polygon = polygon.buffer(0)
        return polygon.wkt if wkt_output else polygon
    except Exception as exc:
        log.debug(
            "Failed to build sector polygon for lat=%s lon=%s radius=%s az=%s beam=%s | %s",
            lat,
            lon,
            radius_m,
            azimuth,
            horizontal_beam,
            exc,
        )
        return None


In [23]:
def normalise_cell_id(
    df: pd.DataFrame,
    technology_col: str = "technology",
    cell_id_col: str = "cell_id",
    enodeb_id_col: str = "enodeb_id",
) -> pd.Series:
    """
    Build the normalised cell identifier used in the Spark version.

    Args:
        df: Input dataframe.
        technology_col: Name of the technology column.
        cell_id_col: Name of the cell_id column.
        enodeb_id_col: Name of the enodeb_id column.

    Returns:
        Pandas Series containing the normalised cell identifier.
    """
    legacy_tech = {"2G", "3G", "3GFEMTO"}

    return df.apply(
        lambda row: row[cell_id_col]
        if row[technology_col] in legacy_tech
        else (row[enodeb_id_col] * 256) + row[cell_id_col],
        axis=1,
    )


In [25]:
def explode_radii(
    df: pd.DataFrame,
    radius_cols: Tuple[str, ...] = ("radii_70", "radii_80", "radii_90"),
) -> pd.DataFrame:
    """
    Explode radius columns into long format with radius_label and radius_m.

    Args:
        df: Input dataframe containing radii columns.
        radius_cols: Tuple of radius column names to unpivot.

    Returns:
        Long-form dataframe with one row per radius type.
    """
    base_cols = [c for c in df.columns if c not in radius_cols]

    frames = []
    for radius_col in radius_cols:
        temp = df[base_cols + [radius_col]].copy()
        temp["radius_label"] = radius_col
        temp["radius_m"] = pd.to_numeric(temp[radius_col], errors="coerce")
        frames.append(temp)

    out_df = pd.concat(frames, ignore_index=True)
    out_df = out_df.loc[out_df["radius_m"].notna() & (out_df["radius_m"] > 0)].copy()

    return out_df


def add_sector_polygons(
    df: pd.DataFrame,
    lat_col: str = "cell_latitude",
    lon_col: str = "cell_longitude",
    radius_col: str = "radius_m",
    azimuth_col: str = "azimuth",
    beam_col: str = "horizontal_beam",
    polygon_col: str = "polygon_coverage",
    n_arc_points: int = 21,
    round_dp: Optional[int] = 4,
) -> pd.DataFrame:
    """
    Add sector polygon WKT to a dataframe.

    Args:
        df: Input dataframe.
        lat_col: Latitude column name.
        lon_col: Longitude column name.
        radius_col: Radius column name.
        azimuth_col: Azimuth column name.
        beam_col: Beam width column name.
        polygon_col: Output polygon column name.
        n_arc_points: Number of points used across the arc.
        round_dp: Optional decimal precision for coordinates.

    Returns:
        Dataframe with polygon column added.
    """
    out_df = df.copy()

    out_df[polygon_col] = out_df.apply(
        lambda row: sector_polygon(
            lat=row[lat_col],
            lon=row[lon_col],
            radius_m=row[radius_col],
            azimuth=row[azimuth_col],
            horizontal_beam=row[beam_col],
            n_arc_points=n_arc_points,
            wkt_output=True,
            round_dp=round_dp,
        ),
        axis=1,
    )

    return out_df


def add_centroids_from_polygon(
    df: pd.DataFrame,
    polygon_col: str = "polygon_coverage",
    cent_lat_col: str = "cell_cent_latitude",
    cent_lon_col: str = "cell_cent_longitude",
) -> pd.DataFrame:
    """
    Fill missing centroid columns from polygon WKT.

    Args:
        df: Input dataframe.
        polygon_col: WKT polygon column name.
        cent_lat_col: Centroid latitude column name.
        cent_lon_col: Centroid longitude column name.

    Returns:
        Dataframe with centroid columns filled where missing.
    """
    out_df = df.copy()

    centroids = out_df[polygon_col].apply(get_centroid)
    centroids_df = pd.DataFrame(centroids.tolist(), columns=["centroid_x", "centroid_y"], index=out_df.index)

    out_df[cent_lat_col] = out_df[cent_lat_col].where(
        out_df[cent_lat_col].notna(),
        centroids_df["centroid_y"],
    )
    out_df[cent_lon_col] = out_df[cent_lon_col].where(
        out_df[cent_lon_col].notna(),
        centroids_df["centroid_x"],
    )

    return out_df


In [26]:
ds = load_dataset(
    "TomFreeman3/cell_service_data",
)

In [27]:
df = ds['train'].to_pandas()

In [30]:

radius_col = "radii_90"

out_df = add_sector_polygons(
    df=df,
    lat_col="cell_latitude",
    lon_col="cell_longitude",
    radius_col=radius_col,
    azimuth_col="azimuth",
    beam_col="horizontal_beam",
    polygon_col="polygon_coverage",
    n_arc_points=21,
    round_dp=4,
)


In [31]:
out_df

,anon_cell_key,cell_latitude,cell_longitude,azimuth,radii_70,radii_80,radii_90,horizontal_beam,polygon_coverage
0,8488048f1157be750e3d726720e000e4f431b39d0142b7...,51.7305,-3.2649,0.0,5700,6200,13200,65.0,"POLYGON ((-3.2649 51.7305, -3.368 51.8305, -3...."
1,6af087648615cf003ddbe622c4dfc7026bddbfc3016546...,53.2348,-3.2012,0.0,800,1000,1200,65.0,"POLYGON ((-3.2012 53.2348, -3.2109 53.2439, -3..."
2,5afe848a8b6221802177d33831cb7d9369e77496625dca...,53.2348,-3.2012,0.0,1000,1100,1400,66.0,"POLYGON ((-3.2012 53.2348, -3.2126 53.2453, -3..."
3,bdf7f9706a5d754f5e33109bf7f8e212517c5791cbb465...,53.2348,-3.2012,0.0,900,900,1000,65.0,"POLYGON ((-3.2012 53.2348, -3.2093 53.2424, -3..."
4,d1c7c68d3bd1a9f10b5bfa466301fb3e0b26698c0f2008...,53.1546,-3.2006,0.0,900,1000,2000,360.0,"POLYGON ((-3.2006 53.1366, -3.2099 53.1375, -3..."
...,...,...,...,...,...,...,...,...,...
25701,58e15f2abb1045d22f5f3510d3f3abdd657d4064b1d1e0...,52.4940,-1.9014,355.0,400,500,600,56.0,"POLYGON ((-1.9014 52.494, -1.9062 52.4985, -1...."
25702,f4548a0c9f2ead8338c636430f1fd991db26dd0167dc43...,52.4940,-1.9014,355.0,400,400,500,64.0,"POLYGON ((-1.9014 52.494, -1.9058 52.4976, -1...."
25703,a1d6a19dcdf521ec1f4c2734db7def6578bf680503b313...,52.4940,-1.9014,355.0,400,500,500,54.0,"POLYGON ((-1.9014 52.494, -1.9053 52.4978, -1...."
25704,790cb748696b3f2d979ab1e294c65b55a09a37c6ba82eb...,52.4940,-1.9014,355.0,600,700,700,61.0,"POLYGON ((-1.9014 52.494, -1.9074 52.4991, -1...."
